# Neural Network: Feed-Forward Network with Cross-Validation

In this notebook, we test whether deep learning can outperform tree-based models on this tabular dataset using K-Fold Cross-Validation for a fair comparison.

1. **Zero Imputation with Flags Strategy** - Special preprocessing for neural networks
2. **ThyroidNet Architecture** - Feed-forward neural network design
3. **Cross-Validation Loop** - 5-fold Stratified CV with PyTorch training
4. **Comparison** - Performance vs tree-based models

## Preprocessing for Neural Networks

Neural networks have different requirements than tree-based models:

- **Cannot handle NaN**: Tensor operations require numeric values
- **Sensitive to scale**: Gradient descent works better with normalized inputs
- **Can learn missingness patterns**: With explicit indicator features

The "Zero Imputation with Flags" strategy addresses these requirements:
- Fill all missing numeric values with 0
- Keep ALL `_measured` flags as features
- Apply standard scaling after imputation

The network learns to interpret the combination of (value=0, measured=False) as "missing" rather than "truly zero".

## Data Loading and Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import copy

from src.data_loader import load_thyroid_data_3_classes
from src.preprocessing import get_zero_imputation_with_flags_pipeline
from src.metrics import thyroid_disease_f2_score

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

/media/DIURNOext4/alejandro/wip-clase/PIA-SAA/example_repos/thyroid/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
X_train_full, X_test, y_train_full, y_test = load_thyroid_data_3_classes(test_size=0.2, random_state=42)

print(f"Training: {len(X_train_full)} samples")
print(f"Test: {len(X_test)} samples (held out)")

Training: 7337 samples
Test: 1835 samples (held out)


In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train_full)
y_test_encoded = label_encoder.transform(y_test)

print(f"Classes: {label_encoder.classes_}")

Classes: ['hyperthyroid' 'hypothyroid' 'negative']


## ThyroidNet Architecture

We design a simple feed-forward network with:

- **Batch Normalization**: Helps with training stability and acts as regularization
- **Dropout**: Prevents overfitting by randomly zeroing activations
- **ReLU activations**: Standard non-linearity for hidden layers
- **Progressively smaller layers**: Feature compression toward the output

In [ ]:
class ThyroidNet(nn.Module):
    
    def __init__(self, input_size: int, num_classes: int = 3, dropout_rate: float = 0.3):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, num_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)

In [ ]:
# Determine number of features using a small sample
preprocessor = get_zero_imputation_with_flags_pipeline()
X_sample = preprocessor.fit_transform(X_train_full.head(5))
n_features = X_sample.shape[1]
n_classes = len(label_encoder.classes_)

model = ThyroidNet(n_features, n_classes)
print(model)

ThyroidNet(
  (network): Sequential(
    (0): Linear(in_features=28, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.3, inplace=False)
    (12): Linear(in_features=32, out_features=3, bias=True)
  )
)


## 5-Fold Cross-Validation Training Loop

In [ ]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
num_epochs = 100
batch_size = 64

fold_recalls = []

print("Starting 5-Fold Cross Validation...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_encoded)):
    print(f"\n{'='*40}")
    print(f"Fold {fold + 1}/{n_splits}")
    print(f"{'='*40}")
    
    # Split data
    X_fold_train = X_train_full.iloc[train_idx]
    y_fold_train = y_train_encoded[train_idx]
    X_fold_val = X_train_full.iloc[val_idx]
    y_fold_val = y_train_encoded[val_idx]
    
    # Preprocess
    preprocessor = get_zero_imputation_with_flags_pipeline()
    X_fold_train_processed = preprocessor.fit_transform(X_fold_train)
    X_fold_val_processed = preprocessor.transform(X_fold_val)
    
    # Create tensors
    X_train_tensor = torch.tensor(X_fold_train_processed.astype(np.float32))
    y_train_tensor = torch.tensor(y_fold_train, dtype=torch.long)
    X_val_tensor = torch.tensor(X_fold_val_processed.astype(np.float32))
    y_val_tensor = torch.tensor(y_fold_val, dtype=torch.long)
    
    # DataLoaders
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Initialize model and optimizer
    model = ThyroidNet(n_features, n_classes)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    best_val_recall = 0
    
    # Training Loop
    for epoch in range(num_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
        # Validation
        model.eval()
        with torch.no_grad():
            val_predictions = []
            val_targets = []
            for X_batch, y_batch in val_loader:
                outputs = model(X_batch)
                predictions = outputs.argmax(dim=1)
                val_predictions.extend(predictions.numpy())
                val_targets.extend(y_batch.numpy())
                
            val_recall = thyroid_disease_f2_score(np.array(val_targets), np.array(val_predictions))
            
            if val_recall > best_val_recall:
                best_val_recall = val_recall
                
    print(f"Best Validation Recall for Fold {fold + 1}: {best_val_recall:.4f}")
    fold_recalls.append(best_val_recall)

mean_recall = np.mean(fold_recalls)
std_recall = np.std(fold_recalls) * 2

print(f"\n{'='*40}")
print(f"Neural Network - Thyroid Mean Recall (5-Fold CV)")
print(f"  Per-fold scores: {np.round(fold_recalls, 3)}")
print(f"  Mean: {mean_recall:.3f} (+/- {std_recall:.3f})")

Starting 5-Fold Cross Validation...

Fold 1/5


Best Validation Recall for Fold 1: 0.8395

Fold 2/5


Best Validation Recall for Fold 2: 0.8309

Fold 3/5


Best Validation Recall for Fold 3: 0.8610

Fold 4/5


Best Validation Recall for Fold 4: 0.8414

Fold 5/5


Best Validation Recall for Fold 5: 0.8281

Neural Network - Thyroid Mean Recall (5-Fold CV)
  Per-fold scores: [0.84  0.831 0.861 0.841 0.828]
  Mean: 0.840 (+/- 0.023)


## Conclusion and Comparison

By utilizing exactly the same Cross-Validation strategy (`StratifiedKFold(n_splits=5)`) and standardizing the test size to `0.2` rather than `0.15`, we achieved a fair comparison between the Neural Network and the advanced tree-based models (XGBoost & CatBoost).

We can now compare the CV Mean Recalls across Notebooks `03` and `04b`. 
Usually, tree-based models excel on small tabular datasets, while Deep Learning shines on larger or unstructured datasets. This experiment verifies whether those heuristics apply directly to the Thyroid dataset.